# Ollama Zero-Shot Regression: Reproducible Development Set

Change the model or prompt settings, rerun the notebook, and compare development MAE. The labeled development rows remain fixed.

In [ ]:
from pathlib import Path
import json
import re
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'whatmakestitle'

# Experiment settings: change these values and rerun from this cell onward.
OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'llama3.2:3b'
DEV_SIZE = 0.20
RANDOM_STATE = 42
TEMPERATURE = 0.0
MAX_TOKENS = 3
REQUEST_TIMEOUT_SECONDS = 120

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
train_idx, dev_idx = train_test_split(
    np.arange(len(train_df)), test_size=DEV_SIZE, random_state=RANDOM_STATE,
    stratify=train_df['interest'])
print(f'Train rows: {len(train_idx)} | dev rows: {len(dev_idx)} | test rows: {len(test_df)}')

## Prompt settings

In [ ]:
SYSTEM_PROMPT = 'Predict online content interest. Return exactly one integer from 1 to 6 and nothing else. 1 is lowest interest and 6 is highest interest. Judge only the title and domain.'

def build_prompt(title, domain):
    return f'{SYSTEM_PROMPT}\n\nTitle: {str(title).strip()}\nDomain: {str(domain).strip()}\n\nInterest score (1-6):'

print(build_prompt('How Hard Is Finding Bigfoot?', 'youtube'))

In [ ]:
def parse_score(response_text):
    match = re.fullmatch(r'\s*([1-6])\s*', response_text)
    if match is None:
        raise ValueError(f'Ollama returned a non-score response: {response_text!r}')
    return int(match.group(1))


def ollama_score(title, domain):
    payload = {
        'model': OLLAMA_MODEL, 'prompt': build_prompt(title, domain),
        'stream': False, 'options': {'temperature': TEMPERATURE, 'num_predict': MAX_TOKENS},
    }
    request = Request(OLLAMA_URL, data=json.dumps(payload).encode('utf-8'), headers={'Content-Type': 'application/json'}, method='POST')
    try:
        with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            result = json.loads(response.read().decode('utf-8'))
    except HTTPError as exc:
        body = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Ollama HTTP {exc.code}: {body}') from exc
    except URLError as exc:
        raise RuntimeError('Could not connect to Ollama. Start Ollama and verify the configured model.') from exc
    if 'response' not in result:
        raise RuntimeError(f'Ollama response did not contain text: {result}')
    return parse_score(result['response'])


def predict_frame(frame):
    predictions = []
    for row_number, row in enumerate(frame.itertuples(index=False), start=1):
        predictions.append(ollama_score(row.title, row.domain))
        if row_number % 10 == 0 or row_number == len(frame):
            print(f'Predicted {row_number}/{len(frame)} rows')
    return np.asarray(predictions, dtype=float)

print(f'Configured model: {OLLAMA_MODEL}')

## Development MAE

In [ ]:
from sklearn.metrics import mean_absolute_error

dev_predictions = predict_frame(train_df.iloc[dev_idx])
dev_actual = train_df.iloc[dev_idx]['interest'].to_numpy(dtype=float)
validation_mae = mean_absolute_error(dev_actual, dev_predictions)
print(f'Development MAE: {validation_mae:.4f}')

In [ ]:
test_predictions = predict_frame(test_df)
submission = pd.DataFrame({'title_id': test_df['title_id'], 'interest': test_predictions})
submission_path = DATA_DIR / 'submission_ollama_zero_shot.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
submission.head()